# Layer 2 — Consumer Wearable Bridge Analysis

Applies the Layer 1 SVM model (trained on Physionet ECG data) to personal Apple Watch
heart rate and HRV data. Quantifies the signal gap between clinical and consumer-grade
sources, generates anomaly probability scores, and tests whether scores are elevated
around the clinical anchor date (June 18, 2025).

**Prerequisite:** `03_modelling.ipynb` must be run first — this notebook loads the saved
SVM model, scaler, and Physionet feature matrix from Layer 1.

**Calls:** `src/apple_watch_features.py` — `build_apple_watch_feature_matrix()`

## Cell 1 — Imports, Paths, Constants, and Layer 1 Artifacts

Loads all dependencies, defines project paths and locked window parameters,
and loads the three Layer 1 artifacts needed for Layer 2 analysis:
- `selected_model.joblib` — fitted SVM classifier
- `scaler.joblib` — fitted StandardScaler (trained on Layer 1 training data only)
- `physionet_features.csv` — Layer 1 feature matrix (for distribution comparison)

**Output paths:**
- `PROC_DIR` → `data/processed/` (for apple_watch_features.csv)
- `LAYER2_DIR` → `outputs/layer2/` (for gap_quantification.csv, probability_scores.csv, mann_whitney_results.csv)

In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy import stats

# ── Project root and src/ access ─────────────────────────────
PROJECT_ROOT = r'C:\Projects\GA Capstone Project'
sys.path.insert(0, PROJECT_ROOT)

from src.apple_watch_features import build_apple_watch_feature_matrix

# ── Input paths ──────────────────────────────────────────────
HR_PATH  = os.path.join(PROJECT_ROOT, 'data', 'processed', 'heart_rate_clean.csv')
HRV_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'hrv_clean.csv')

# ── Output paths ─────────────────────────────────────────────
PROC_DIR   = os.path.join(PROJECT_ROOT, 'data', 'processed')
LAYER2_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'layer2')
FIGURE_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'figures')

os.makedirs(LAYER2_DIR, exist_ok=True)

# ── Locked window parameters ─────────────────────────────────
WINDOW_SIZE_MINUTES = 30
STEP_SIZE_MINUTES   = 15
MIN_READINGS        = 10

# ── Locked feature column order ──────────────────────────────
FEATURE_COLS = [
    'rmssd', 'sdnn', 'mean_rr', 'pnn50',
    'hr_mean', 'hr_std', 'rr_skewness', 'rr_kurtosis'
]

# ── Load Layer 1 artifacts ───────────────────────────────────
MODEL_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'models')

scaler = joblib.load(os.path.join(MODEL_DIR, 'scaler.joblib'))
model  = joblib.load(os.path.join(MODEL_DIR, 'selected_model.joblib'))

physionet_features = pd.read_csv(
    os.path.join(PROC_DIR, 'physionet_features.csv')
)

# ── Confirmation ─────────────────────────────────────────────
print('Layer 1 artifacts loaded:')
print(f'  Physionet features shape: {physionet_features.shape}')
print(f'  Scaler type:             {type(scaler).__name__}')
print(f'  Model type:              {type(model).__name__}')
print(f'  LAYER2_DIR exists:       {os.path.isdir(LAYER2_DIR)}')

Layer 1 artifacts loaded:
  Physionet features shape: (8187, 11)
  Scaler type:             StandardScaler
  Model type:              SVC
  LAYER2_DIR exists:       True


## Cell 2 — Build Apple Watch Feature Matrix

Calls `build_apple_watch_feature_matrix()` with locked window parameters
(30 min window, 15 min step, min 10 readings). The function extracts HR and HRV
features, then merges them using a vectorised interval join — only windows with
both sufficient HR readings and at least one HRV record are retained.

Validates the resulting matrix for missing values before saving to
`data/processed/apple_watch_features.csv`.

**Calls:** `src/apple_watch_features.py` — `build_apple_watch_feature_matrix()`
**Output:** `data/processed/apple_watch_features.csv`

In [2]:
# Calls: src/apple_watch_features.build_apple_watch_feature_matrix()
# Input: data/processed/heart_rate_clean.csv, data/processed/hrv_clean.csv
# Output: data/processed/apple_watch_features.csv

aw_features = build_apple_watch_feature_matrix(
    hr_path=HR_PATH,
    hrv_path=HRV_PATH,
    window_size_minutes=WINDOW_SIZE_MINUTES,
    step_size_minutes=STEP_SIZE_MINUTES,
    min_readings=MIN_READINGS
)

# ── Missing value check ─────────────────────────────────────
missing = aw_features[FEATURE_COLS].isnull().sum().sum()
print(f"\nMissing values across 8 features: {missing}")

if missing > 0:
    null_counts = aw_features[FEATURE_COLS].isnull().sum()
    print("\nWARNING — features with missing values:")
    print(null_counts[null_counts > 0])
    raise ValueError(
        f"Feature matrix contains {missing} missing values. "
        "Cannot save — investigate before proceeding."
    )

# ── Save ─────────────────────────────────────────────────────
aw_save_path = os.path.join(PROC_DIR, 'apple_watch_features.csv')
aw_features.to_csv(aw_save_path, index=False)
print(f"\nSaved: {aw_save_path}")

# ── Summary ──────────────────────────────────────────────────
print(f"\nApple Watch feature matrix shape: {aw_features.shape}")
print(f"Date range: {aw_features['window_start'].min()} → {aw_features['window_end'].max()}")
print(f"\nAnchor period distribution:")
print(aw_features['anchor_period'].value_counts().sort_index())
print()
aw_features.head()

C:\Projects\GA Capstone Project\src\apple_watch_features.py:110: DtypeWarning: Columns (0: sourceVersion) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(hr_path, parse_dates=['startDate'])


Apple Watch feature matrix built:
  HR windows produced:      12,352
  HRV records available:    5,456
  Windows after merge:      1,997
  Windows dropped (no HRV): 10,355

Missing values across 8 features: 0

Saved: C:\Projects\GA Capstone Project\data\processed\apple_watch_features.csv

Apple Watch feature matrix shape: (1997, 12)
Date range: 2021-04-26 11:06:37+08:00 → 2026-02-17 07:36:37+08:00

Anchor period distribution:
anchor_period
baseline       1758
follow_up       115
post_anchor      29
pre_anchor       95
Name: count, dtype: int64



,window_start,window_end,rmssd,sdnn,mean_rr,pnn50,hr_mean,hr_std,rr_skewness,rr_kurtosis,n_readings,anchor_period
0,2021-04-26 11:06:37+08:00,2021-04-26 11:36:37+08:00,36.2728,42.6739,499.4833,0.0370,121.4359,11.2161,3.2992,13.0293,82,baseline
1,2021-04-26 15:36:37+08:00,2021-04-26 16:06:37+08:00,20.6874,24.3381,581.4489,0.1163,103.5996,6.6254,0.0363,-0.2358,44,baseline
2,2021-04-27 10:06:37+08:00,2021-04-27 10:36:37+08:00,33.4710,39.3776,499.7781,0.0690,121.2081,11.1536,1.8704,4.1548,59,baseline
3,2021-04-27 10:21:37+08:00,2021-04-27 10:51:37+08:00,33.4710,39.3776,554.3361,0.0732,108.8161,7.6622,1.6243,2.9349,42,baseline
4,2021-04-28 02:21:37+08:00,2021-04-28 02:51:37+08:00,38.2412,44.9896,957.7325,0.3333,62.7811,3.0539,0.1634,0.7834,10,baseline


In [3]:
import pandas as pd

hr = pd.read_csv(
    r'C:\Projects\GA Capstone Project\data\processed\heart_rate_clean.csv',
    parse_dates=['startDate']
)
hrv = pd.read_csv(
    r'C:\Projects\GA Capstone Project\data\processed\hrv_clean.csv',
    parse_dates=['startDate']
)

anchor_date = pd.Timestamp('2025-06-18', tz='Asia/Singapore')
post_start  = anchor_date
post_end    = anchor_date + pd.Timedelta(days=90)

hr_post  = hr[(hr['startDate'] >= post_start) & (hr['startDate'] < post_end)]
hrv_post = hrv[(hrv['startDate'] >= post_start) & (hrv['startDate'] < post_end)]

print("Post-Anchor Period Raw Data")
print("="*45)
print(f"HR readings:  {len(hr_post):,}")
print(f"HRV records:  {len(hrv_post):,}")
print(f"\nHRV records per week in post_anchor:")
hrv_post_copy = hrv_post.copy()
hrv_post_copy['week'] = hrv_post_copy['startDate'].dt.isocalendar().week
print(hrv_post_copy.groupby('week').size().to_string())

C:\Users\lucas\AppData\Local\Temp\ipykernel_9392\547706390.py:3: DtypeWarning: Columns (0: sourceVersion) have mixed types. Specify dtype option on import or set low_memory=False.
  hr = pd.read_csv(


Post-Anchor Period Raw Data
HR readings:  12,383
HRV records:  48

HRV records per week in post_anchor:
week
25    3
26    1
27    3
28    2
29    4
30    7
31    3
32    4
33    2
34    1
35    4
36    6
37    7
38    1
